# Paper 2 — ResNet-50 hybrid SRAM/PCM-IMC accuracy (ImageNet, H100)

Same IMC model as the LeNet experiment (cellBit=2 sign-magnitude, two's-complement bit-serial input, 64-row tiling, 5-bit ADC, per-weight drift, per-cell write noise; protected top-k exact).

Validated in sandbox: the IMC forward matches a direct-INT8 reference **bit-exactly** at no-drift/no-ADC/no-noise. Here we run the accuracy sweep on a fixed random subset of ImageNet val.

Upload **only this notebook**. It reads `kaggle.json` and `LayerAware_Ratios_ResNet50.json` from your Drive `MyDrive/PCM_Research7`.

## 0. GPU check (expect H100)

In [1]:
!nvidia-smi

Thu Jun 11 14:14:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   25C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/PCM_Research7'
import os; print('exists:', os.path.exists(PROJECT)); print(os.listdir(PROJECT)[:10])

Mounted at /content/drive
exists: True
['PCM_Research5_Data_Step3_Sensitivity.csv', 'PCM_Research6_Data_Step3_Sensitivity.csv', 'Step_B6_PrecisionAblation.py', 'README.md', 'Step_B2_BudgetSweep.py', 'Step_B1_LayerSensitivity.py', 'Step_B3_LongTerm.py', 'Step_B4_Mismatch.py', 'Step_B1_Aggregate.py', 'config_imagenet.py']


## 2. Download ImageNet val (Kaggle, 6.7 GB) — same as your Module-B notebook

In [3]:
# Kaggle credentials from your Drive PCM_Research7
import os, time, glob
!mkdir -p ~/.kaggle
!cp {PROJECT}/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!pip install kaggle -q

VAL_ROOT = '/content/imagenet_val'
need = (not os.path.exists(VAL_ROOT) or
        len([d for d in os.listdir(VAL_ROOT) if d.startswith('n0')]) < 100)
if need:
    !mkdir -p {VAL_ROOT}
    t0=time.time(); print('Downloading titericz/imagenet1k-val ...')
    !kaggle datasets download -d titericz/imagenet1k-val -p /content --quiet
    print(f'Downloaded {time.time()-t0:.0f}s'); print('Unzipping ...'); t0=time.time()
    !unzip -q /content/imagenet1k-val.zip -d {VAL_ROOT}
    !rm /content/imagenet1k-val.zip
    print(f'Unzipped {time.time()-t0:.0f}s')
else:
    print('Val set already present.')

cand = glob.glob(f'{VAL_ROOT}/**/n01440764', recursive=True)
VAL_ROOT = os.path.dirname(cand[0]) if cand else VAL_ROOT
syn = [d for d in os.listdir(VAL_ROOT) if d.startswith('n0')]
print('VAL_ROOT =', VAL_ROOT, '| synset dirs:', len(syn), '(expect 1000)')

Dataset URL: https://www.kaggle.com/datasets/titericz/imagenet1k-val
License(s): CC0-1.0
Downloaded 46s
Unzipping ...
Unzipped 25s
VAL_ROOT = /content/imagenet_val/imagenet-val | synset dirs: 982 (expect 1000)


## 3. Write the validated IMC code to disk

In [4]:
%%writefile imc_torch.py
#!/usr/bin/env python3
"""
imc_torch.py -- torch (GPU-capable) port of imc_vec.py's INT8 hybrid SRAM/PCM-IMC
forward for LeNet. Same math as the numpy reference (imc_vec.py); runs on cuda for
speed on Colab. Validated to reproduce imc_vec predictions image-for-image, and to
match the RTL golden bit-exactly at the no-drift/no-ADC/no-noise point.

dtype: float64 reproduces numpy exactly; float32 is fast on GPU and is exact for
LeNet layer sizes (all integer accumulations stay below 2^24). For deeper nets
(ImageNet) use float64 or int accumulation.
"""

import math
import torch
import numpy as np
from imc_vec import load_model, read_hex_bytes, LAYERS, WCOUNT  # reuse numpy model loader

DEFAULT_CFG = {
    "cellBit": 2, "n_mag_cells": 4, "subArray": 64, "in_bits": 8,
    "adc_bits": None, "drift_t": 1.0, "comp": 0.0,
}


# ---------- move model to torch + precompute cells ----------
def to_torch_model(model, device="cpu", dtype=torch.float32, cellBit=2, ncell=4):
    cmax = (1 << cellBit) - 1
    tm = {"rshift": model["rshift"], "W": {}, "mask": {}, "nu": {}, "bias": {}, "cells": {}}
    for L in LAYERS:
        W = model["W"][L]                                  # numpy int (out,K)
        absW = np.abs(W)
        cells = np.stack([((absW >> (cellBit * j)) & cmax) for j in range(ncell)], axis=-1)
        tm["W"][L] = torch.tensor(W, device=device, dtype=dtype)
        tm["mask"][L] = torch.tensor(model["mask"][L], device=device)        # bool
        tm["nu"][L] = torch.tensor(model["nu"][L], device=device, dtype=dtype)
        tm["bias"][L] = torch.tensor(model["bias"][L], device=device, dtype=dtype)
        tm["cells"][L] = torch.tensor(cells, device=device, dtype=dtype)     # (out,K,ncell)
    return tm


def writenoise_torch(model, seed, device="cpu", dtype=torch.float32, cellBit=2, ncell=4):
    sigma_value = 0.03 * 127.0
    S = sum((2 ** (cellBit * j)) ** 2 for j in range(ncell))
    sc = sigma_value / math.sqrt(S)
    rng = np.random.RandomState(seed)
    return {L: torch.tensor(rng.normal(0.0, sc, model["W"][L].shape + (ncell,)),
                            device=device, dtype=dtype) for L in LAYERS}


def nu_torch(model, seed, device="cpu", dtype=torch.float32):
    rng = np.random.RandomState(seed)
    return {L: torch.tensor(rng.normal(0.10, 0.03, model["W"][L].shape),
                            device=device, dtype=dtype) for L in LAYERS}


# ---------- helpers ----------
def adc_quant_t(x, fs, bits):
    levels = (1 << bits) - 1
    safe = torch.where(fs > 0, fs, torch.ones_like(fs))
    q = torch.where(fs > 0, torch.round(x / safe * levels), torch.zeros_like(x))
    q = torch.clamp(q, 0, levels)
    return q / levels * fs


def requant_relu_t(acc, rshift):
    shifted = torch.floor(acc / (1 << rshift))
    return torch.where(acc <= 0, torch.zeros_like(acc), torch.clamp(shifted, max=127))


def requant_fc3_t(acc, rshift):
    shifted = torch.floor(acc / (1 << rshift))
    POS, NEG = 32767, -131071
    tr = shifted.to(torch.int64) & 0xFFFF
    tr = torch.where((tr & 0x8000) != 0, tr - 0x10000, tr).to(acc.dtype)
    res = torch.where(shifted > POS, torch.full_like(shifted, 32767.0),
                      torch.where(shifted < NEG, torch.full_like(shifted, -32768.0), tr))
    return res


def im2col_t(x, kh, kw, pad):
    B, C, H, W = x.shape
    OH, OW = H + 2 * pad - kh + 1, W + 2 * pad - kw + 1
    if pad:
        xp = torch.zeros((B, C, H + 2 * pad, W + 2 * pad), device=x.device, dtype=x.dtype)
        xp[:, :, pad:pad + H, pad:pad + W] = x
    else:
        xp = x
    cols = torch.empty((B, C, kh, kw, OH, OW), device=x.device, dtype=x.dtype)
    for ky in range(kh):
        for kx in range(kw):
            cols[:, :, ky, kx, :, :] = xp[:, :, ky:ky + OH, kx:kx + OW]
    return cols.reshape(B, C * kh * kw, OH * OW), OH, OW


# ---------- IMC matmul (torch) ----------
def imc_matmul_t(tm, L, V, cfg, kind, wn_L=None):
    cells_L, W_L, mask_L, nu_L, bias_L = tm["cells"][L], tm["W"][L], tm["mask"][L], tm["nu"][L], tm["bias"][L]
    rshift = tm["rshift"][L]
    device, dtype = W_L.device, W_L.dtype
    out_dim, K = W_L.shape
    cellBit, ncell, sub, in_bits = cfg["cellBit"], cfg["n_mag_cells"], cfg["subArray"], cfg["in_bits"]
    adc_bits, t, comp = cfg["adc_bits"], cfg["drift_t"], cfg["comp"]
    cmax = (1 << cellBit) - 1
    M = V.shape[1]
    zeroW = torch.zeros_like(W_L)

    Wp = torch.where(mask_L, W_L, zeroW)
    acc = bias_L[:, None] + Wp @ V

    Wu = torch.where(mask_L, zeroW, W_L)
    if not bool((Wu != 0).any()):
        # fully protected layer (e.g. conv1): no PCM contribution, skip the analog path
        if kind == "raw":
            return acc
        if kind == "fc3":
            return requant_fc3_t(acc, rshift)
        return requant_relu_t(acc, rshift)
    signpos = (Wu > 0).to(dtype)
    signneg = (Wu < 0).to(dtype)
    if t != 1.0:
        Dpcm = torch.pow(torch.as_tensor(float(t), device=device, dtype=dtype), -nu_L)
        D = torch.where(mask_L, torch.ones_like(nu_L), Dpcm)
    else:
        D = torch.ones_like(W_L)

    T = (K + sub - 1) // sub
    Kpad = T * sub
    actual_rows = torch.full((T,), float(sub), device=device, dtype=dtype)
    if Kpad != K:
        actual_rows[-1] = float(K - (T - 1) * sub)

    def padK(a):
        if Kpad == K:
            return a
        o = torch.zeros(a.shape[:-1] + (Kpad,), device=device, dtype=dtype)
        o[..., :K] = a
        return o

    Vpad = torch.zeros((Kpad, M), device=device, dtype=dtype)
    Vpad[:K, :] = V
    Vint = Vpad.to(torch.int64)
    Vbits = torch.stack([((Vint >> i) & 1).to(dtype) for i in range(in_bits)], dim=0)
    Vbits_t = Vbits.reshape(in_bits, T, sub, M)

    notmask = (~mask_L).to(dtype)
    pcm = torch.zeros((out_dim, M), device=device, dtype=dtype)
    for j in range(ncell):
        Cj = cells_L[:, :, j]
        if wn_L is not None:
            Cj = Cj + wn_L[:, :, j] * notmask
        CjD_pos = padK(Cj * D * signpos).reshape(out_dim, T, sub)
        CjD_neg = padK(Cj * D * signneg).reshape(out_dim, T, sub)
        wj = float(1 << (cellBit * j))
        for i in range(in_bits):
            Bi = Vbits_t[i]
            ppos = torch.einsum('trm,otr->otm', Bi, CjD_pos)
            pneg = torch.einsum('trm,otr->otm', Bi, CjD_neg)
            if adc_bits is not None:
                fs = (actual_rows * cmax)
                ppos = adc_quant_t(ppos, fs[None, :, None], adc_bits)
                pneg = adc_quant_t(pneg, fs[None, :, None], adc_bits)
            contrib = (ppos - pneg).sum(dim=1)
            bw = -(1 << i) if i == in_bits - 1 else (1 << i)
            pcm = pcm + (float(bw) * wj) * contrib
    if comp != 0.0:
        pcm = pcm * (math.exp(comp * math.log(t)) if t > 0 else 1.0)
    acc = acc + pcm

    if kind == "raw":
        return acc
    if kind == "fc3":
        return requant_fc3_t(acc, rshift)
    return requant_relu_t(acc, rshift)


# ---------- full LeNet forward (torch, batched) ----------
def forward_batch_t(images_int, tm, cfg, wn=None):
    device, dtype = tm["W"]["conv1"].device, tm["W"]["conv1"].dtype
    B = images_int.shape[0]
    gw = lambda L: (wn[L] if wn is not None else None)
    x = torch.tensor(images_int.reshape(B, 1, 28, 28), device=device, dtype=dtype)

    col, OH, OW = im2col_t(x, 5, 5, 2)
    V = col.permute(1, 0, 2).reshape(25, B * OH * OW)
    c1 = imc_matmul_t(tm, "conv1", V, cfg, "relu", gw("conv1"))
    c1 = c1.reshape(6, B, OH * OW).permute(1, 0, 2).reshape(B, 6, 28, 28)
    p1 = c1.reshape(B, 6, 14, 2, 14, 2).amax(dim=(3, 5))

    col, OH, OW = im2col_t(p1, 5, 5, 0)
    V = col.permute(1, 0, 2).reshape(150, B * OH * OW)
    c2 = imc_matmul_t(tm, "conv2", V, cfg, "relu", gw("conv2"))
    c2 = c2.reshape(16, B, OH * OW).permute(1, 0, 2).reshape(B, 16, 10, 10)
    p2 = c2.reshape(B, 16, 5, 2, 5, 2).amax(dim=(3, 5))
    flat = p2.reshape(B, 400)

    f1 = imc_matmul_t(tm, "fc1", flat.t().contiguous(), cfg, "relu", gw("fc1")).t()
    f2 = imc_matmul_t(tm, "fc2", f1.t().contiguous(), cfg, "relu", gw("fc2")).t()
    f3 = imc_matmul_t(tm, "fc3", f2.t().contiguous(), cfg, "fc3", gw("fc3")).t()
    preds = torch.argmax(f3, dim=1)
    return f3, preds


Writing imc_torch.py


In [5]:
%%writefile imc_resnet.py
#!/usr/bin/env python3
"""
imc_resnet.py -- ResNet-50 hybrid SRAM/PCM-IMC forward (analog crossbar + ADC).

Each PCM-mapped conv/linear (per config_imagenet.is_pcm_weight: weight, not BN,
not downsample.1) is replaced by an IMC module that:
  - quantizes its input activations to INT8 (per-layer calibrated scale),
  - runs the same validated IMC MAC as LeNet (imc_torch.imc_matmul_t, "raw"):
    cellBit=2 sign-magnitude cells, two's-complement bit-serial input, 64-row
    subarray tiling, 5-bit ADC per tile, per-weight drift t^(-nu), per-cell write
    noise; protected (top-k) weights computed exactly (SRAM),
  - dequantizes the integer MAC by (act_scale * weight_scale) to float; BN/ReLU/
    residual/pool stay in float (weight-only-quant baseline keeps activations fp,
    so between IMC convs we return to float exactly as that pipeline does).

mode='ref' replaces the IMC MAC with an exact INT8 conv (F.conv2d on the integer
tensors) so the IMC path can be validated bit-exactly at no-drift/no-ADC/no-noise.

Weights are quantized exactly as config_imagenet.build_quant_cache:
scale = w.abs().max()/127, w_q = round(w/scale). Protection top-k is by |w_orig|,
k = round(ratio*numel), matching config_imagenet.apply_drift_inplace.
"""

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from imc_torch import imc_matmul_t

CELLBIT, NCELL = 2, 4
DRIFT_MEAN, DRIFT_STD, WRITE_NOISE = 0.10, 0.03, 0.03
BASE_CFG = {"cellBit": CELLBIT, "n_mag_cells": NCELL, "subArray": 64, "in_bits": 8,
            "adc_bits": None, "drift_t": 1.0, "comp": 0.0}


def writenoise_sigma_cell(cellBit=CELLBIT, ncell=NCELL):
    return WRITE_NOISE * 127.0 / math.sqrt(sum((2 ** (cellBit * j)) ** 2 for j in range(ncell)))


def is_pcm_weight(name):
    if "weight" not in name:
        return False
    if "bn" in name or "downsample.1" in name:
        return False
    return True


def _cells_from_wq(Wq2d, device, dtype):
    absW = Wq2d.abs().to(torch.int64)
    cmax = (1 << CELLBIT) - 1
    return torch.stack([((absW >> (CELLBIT * j)) & cmax).to(dtype)
                        for j in range(NCELL)], dim=-1)


class _IMCBase(nn.Module):
    """Holds INT8 weight + cells + per-run state shared by conv and linear."""
    def __init__(self, weight, device, dtype):
        super().__init__()
        self.device_, self.dtype_ = device, dtype
        w = weight.detach().to(torch.float32)
        self.w_orig = w.clone()
        wmax = w.abs().max()
        self.w_scale = float(wmax / 127.0) if wmax > 0 else 1.0
        wq4d = torch.round(w / (self.w_scale + 1e-9))                 # (out, ...)
        self.out_ch = wq4d.shape[0]
        self.Wq4d = wq4d.to(device=device, dtype=dtype)              # for ref conv
        self.Wq2d = wq4d.reshape(self.out_ch, -1).to(device=device, dtype=dtype)
        self.cells = _cells_from_wq(self.Wq2d, device, dtype)        # (out,K,ncell)
        self.numel = int(self.w_orig.numel())
        # per-run state
        self.act_scale = 1.0
        self.mask = None
        self.nu = None
        self.wn = None
        self.cfg = dict(BASE_CFG)
        self.mode = "imc"
        self.bias0 = torch.zeros(self.out_ch, device=device, dtype=dtype)

    def _mini_tm(self):
        return {"cells": {"x": self.cells}, "W": {"x": self.Wq2d}, "mask": {"x": self.mask},
                "nu": {"x": self.nu}, "bias": {"x": self.bias0}, "rshift": {"x": 0}}

    def set_state(self, act_scale, mask, nu, wn, cfg, mode="imc"):
        self.act_scale = float(act_scale)
        self.mask = mask
        self.nu = nu
        self.wn = wn
        self.cfg = cfg
        self.mode = mode


class IMCConv2d(_IMCBase):
    def __init__(self, conv, device, dtype):
        super().__init__(conv.weight, device, dtype)
        self.stride = conv.stride
        self.padding = conv.padding
        self.dilation = conv.dilation
        self.kh, self.kw = conv.kernel_size

    def forward(self, x):
        B, Cin, H, W = x.shape
        sH, sW = self.stride; pH, pW = self.padding; dH, dW = self.dilation
        OH = (H + 2 * pH - dH * (self.kh - 1) - 1) // sH + 1
        OW = (W + 2 * pW - dW * (self.kw - 1) - 1) // sW + 1
        xq = torch.round(x / (self.act_scale + 1e-9)).clamp(-128, 127).to(self.dtype_)
        deq = self.act_scale * self.w_scale
        if self.mode == "ref":
            out = F.conv2d(xq, self.Wq4d, None, self.stride, self.padding, self.dilation)
            return out * deq
        V = F.unfold(xq, (self.kh, self.kw), self.dilation, self.padding, self.stride)  # (B,K,L)
        K = V.shape[1]
        Vk = V.permute(1, 0, 2).reshape(K, B * OH * OW)
        acc = imc_matmul_t(self._mini_tm(), "x", Vk, self.cfg, "raw", self.wn)          # (out, B*L)
        out = acc.reshape(self.out_ch, B, OH * OW).permute(1, 0, 2).reshape(B, self.out_ch, OH, OW)
        return out * deq


class IMCLinear(_IMCBase):
    def __init__(self, lin, device, dtype):
        super().__init__(lin.weight, device, dtype)
        self.lin_bias = None if lin.bias is None else lin.bias.detach().to(device=device, dtype=dtype)

    def forward(self, x):
        xq = torch.round(x / (self.act_scale + 1e-9)).clamp(-128, 127).to(self.dtype_)
        deq = self.act_scale * self.w_scale
        if self.mode == "ref":
            out = xq @ self.Wq2d.t()
        else:
            acc = imc_matmul_t(self._mini_tm(), "x", xq.t().contiguous(), self.cfg, "raw", self.wn)
            out = acc.t()
        out = out * deq
        if self.lin_bias is not None:
            out = out + self.lin_bias
        return out


# ---------- module tree surgery ----------
def _get_parent(model, path):
    parts = path.split(".")
    obj = model
    for p in parts[:-1]:
        obj = obj[int(p)] if p.isdigit() else getattr(obj, p)
    return obj, parts[-1]


def _set_module(model, path, new):
    parent, last = _get_parent(model, path)
    if last.isdigit():
        parent[int(last)] = new
    else:
        setattr(parent, last, new)


def build_imc_resnet(model, device="cuda", dtype=torch.float32):
    """Swap every PCM-mapped conv/linear with an IMC module. Returns {path: imc_module}."""
    imc = {}
    pcm_names = [n for n, _ in model.named_parameters() if is_pcm_weight(n)]
    mods = dict(model.named_modules())
    for n in pcm_names:
        path = n[:-len(".weight")]
        m = mods[path]
        new = IMCLinear(m, device, dtype) if isinstance(m, nn.Linear) else IMCConv2d(m, device, dtype)
        _set_module(model, path, new)
        imc[path] = new
    model.to(device).eval()
    return imc


# ---------- activation-scale calibration ----------
def calibrate_act_scales(imc_modules, model, calib_batches, device):
    """Run a few fp batches in 'ref' mode and record per-layer input max-abs.
    (ref mode quantizes with the CURRENT scale but we only need the input range;
    we set scale=1 first so xq=round(x) does not clip the range estimate.)"""
    maxabs = {p: 0.0 for p in imc_modules}
    handles = []
    for p, m in imc_modules.items():
        def hook(mod, inp, _p=p):
            v = inp[0].detach().abs().max().item()
            if v > maxabs[_p]:
                maxabs[_p] = v
        handles.append(m.register_forward_pre_hook(hook))
    # temporary identity-ish state so the model runs (ref mode, scale 1, no clip issue
    # because ref path computes F.conv2d on round(x); fine for range collection)
    for m in imc_modules.values():
        m.set_state(1.0, torch.zeros(m.Wq2d.shape, dtype=torch.bool, device=device),
                    None, None, dict(BASE_CFG), mode="ref")
    with torch.no_grad():
        for xb in calib_batches:
            model(xb.to(device))
    for h in handles:
        h.remove()
    return {p: (maxabs[p] / 127.0 if maxabs[p] > 0 else 1.0) for p in imc_modules}


# ---------- protection masks (top-k by |w_orig|, matching config_imagenet) ----------
def make_masks(imc_modules, ratios, device):
    masks = {}
    for p, m in imc_modules.items():
        r = float(ratios.get(p + ".weight", ratios.get(p, 0.0)))
        k = max(0, min(int(round(r * m.numel)), m.numel))
        flat = m.w_orig.abs().flatten()
        mk = torch.zeros(m.numel, dtype=torch.bool)
        if k > 0:
            idx = torch.topk(flat, k, largest=True, sorted=False).indices
            mk[idx] = True
        masks[p] = mk.view(m.Wq2d.shape).to(device)
    return masks


def uniform_ratios(imc_modules, budget=0.05):
    return {p: budget for p in imc_modules}


def none_ratios(imc_modules):
    return {p: 0.0 for p in imc_modules}


# ---------- per-seed drift / write-noise ----------
def gen_nu(imc_modules, seed, device, dtype):
    rng = np.random.RandomState(seed)
    return {p: torch.tensor(rng.normal(DRIFT_MEAN, DRIFT_STD, m.Wq2d.shape),
                            device=device, dtype=dtype) for p, m in imc_modules.items()}


def gen_wn(imc_modules, seed, device, dtype):
    sc = writenoise_sigma_cell()
    rng = np.random.RandomState(seed)
    return {p: torch.tensor(rng.normal(0.0, sc, m.Wq2d.shape + (NCELL,)),
                            device=device, dtype=dtype) for p, m in imc_modules.items()}


def set_run_state(imc_modules, act_scales, masks, nu, wn, drift_t, adc_bits, comp, mode="imc"):
    cfg = dict(BASE_CFG)
    cfg["drift_t"] = drift_t; cfg["adc_bits"] = adc_bits; cfg["comp"] = comp
    for p, m in imc_modules.items():
        m.set_state(act_scales[p], masks[p], nu[p] if nu else None,
                    wn[p] if wn else None, cfg, mode=mode)


Writing imc_resnet.py


In [6]:
%%writefile run_resnet_imc.py
#!/usr/bin/env python3
"""
run_resnet_imc.py -- ResNet-50 hybrid SRAM/PCM-IMC accuracy sweep on an ImageNet
val subset (H100). Same IMC model as the LeNet experiment (cellBit=2 sign-magnitude,
two's-complement bit-serial input, 64-row tiling, 5-bit ADC, per-weight drift,
per-cell write noise; protected top-k computed exactly).

Arms: none / uniform(beta=5%) / layer_aware(Table IX ratios).
For each arm: a 'clean' point (full INT8, no drift/ADC/noise, via the exact-INT8
ref path) plus a sweep over {no-ADC, 5-bit ADC} x drift-grid x seeds.

The storage-drift Paper-1 baseline (76.14%) is weight-only INT8 with fp activations;
this experiment is full INT8 (weights AND activations, since the IMC crossbar needs
a digital input), so the clean point sits a little below 76.14% by the activation
-quant penalty. The comparison that matters is within this experiment, across arms.

Subset is a fixed seeded random sample of the 50000 val images (deterministic).
Run --timing_only first to size n_images / drift_grid / seeds for the full run.
"""

import argparse, json, time, platform
import numpy as np
import torch

import imc_resnet as R


# ---------------- core (model/loader-agnostic, unit-testable) ----------------
def float_calibrate(model, pcm_paths, loader, device, n_batches):
    """Per-layer input-activation scale = max-abs / 127, measured on the FLOAT
    model (before IMC swap) over a few batches. This is the activation
    quantization scale the IMC crossbar input uses."""
    import torch.nn as nn
    mods = dict(model.named_modules())
    maxabs = {p: 0.0 for p in pcm_paths}
    handles = []
    for p in pcm_paths:
        def hook(mod, inp, _p=p):
            v = inp[0].detach().abs().max().item()
            if v > maxabs[_p]:
                maxabs[_p] = v
        handles.append(mods[p].register_forward_pre_hook(hook))
    model.eval()
    with torch.no_grad():
        for b, (x, _) in enumerate(loader):
            model(x.to(device))
            if b + 1 >= n_batches:
                break
    for h in handles:
        h.remove()
    return {p: (maxabs[p] / 127.0 if maxabs[p] > 0 else 1.0) for p in pcm_paths}


@torch.no_grad()
def eval_loader(model, loader, device):
    c1 = c5 = tot = 0
    for x, y in loader:
        x = x.to(device); y = y.to(device)
        out = model(x)
        _, p5 = out.topk(5, 1, True, True)
        corr = p5.eq(y.view(-1, 1))
        c1 += corr[:, 0].sum().item()
        c5 += corr.any(1).sum().item()
        tot += y.size(0)
    return c1 / tot, c5 / tot


def build_arms(imc, ratios, device, budget=0.05):
    return {
        "none":        R.make_masks(imc, R.none_ratios(imc), device),
        "uniform":     R.make_masks(imc, R.uniform_ratios(imc, budget), device),
        "layer_aware": R.make_masks(imc, ratios, device),
    }


def run_config(model, imc, loader, device, dt, act_scales, mask,
               drift_t, adc_bits, seed, wn_on):
    """One full-subset evaluation under a given (mask, drift_t, adc, seed)."""
    nu = R.gen_nu(imc, 1000 + seed, device, dt) if drift_t != 1.0 else None
    wn = R.gen_wn(imc, 5000 + seed, device, dt) if wn_on else None
    comp = R.DRIFT_MEAN if drift_t != 1.0 else 0.0
    R.set_run_state(imc, act_scales, mask, nu, wn, drift_t, adc_bits, comp, mode="imc")
    return eval_loader(model, loader, device)


def sweep(model, imc, loader, device, dt, act_scales, arms,
          drift_grid, seeds, wn_on, verbose=True):
    runs = []
    # clean per arm: exact INT8 (ref path), no drift / ADC / noise
    for arm, mask in arms.items():
        R.set_run_state(imc, act_scales, mask, None, None, 1.0, None, 0.0, mode="ref")
        a1, a5 = eval_loader(model, loader, device)
        runs.append({"arm": arm, "kind": "clean", "drift_t": 1.0, "adc_bits": None,
                     "top1": a1, "top5": a5})
        if verbose:
            print(f"  clean {arm:12s} top1={a1:.4f} top5={a5:.4f}", flush=True)
    # sweep: arm x adc x drift x seed
    for arm, mask in arms.items():
        for adc in (None, 5):
            for t in drift_grid:
                t1s, t5s = [], []
                for s in range(seeds):
                    a1, a5 = run_config(model, imc, loader, device, dt, act_scales,
                                        mask, t, adc, s, wn_on)
                    t1s.append(a1); t5s.append(a5)
                rec = {"arm": arm, "kind": "sweep", "drift_t": t,
                       "adc_bits": adc, "seeds": list(range(seeds)),
                       "top1_seeds": t1s, "top5_seeds": t5s,
                       "top1_mean": float(np.mean(t1s)), "top1_std": float(np.std(t1s)),
                       "top5_mean": float(np.mean(t5s)), "top5_std": float(np.std(t5s))}
                runs.append(rec)
                if verbose:
                    print(f"  {arm:12s} adc={str(adc):4s} t={t:>8.0f} "
                          f"top1={rec['top1_mean']:.4f}±{rec['top1_std']:.4f}", flush=True)
    return runs


# ---------------- ImageNet wiring (used on Colab; not exercised in sandbox) ---
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def imagenet_subset_loader(val_root, n_images, subset_seed, batch_size, num_workers):
    from torchvision import datasets, transforms
    tx = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                             transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    ds = datasets.ImageFolder(val_root, transform=tx)
    rng = np.random.RandomState(subset_seed)
    idx = np.sort(rng.choice(len(ds), size=min(n_images, len(ds)), replace=False))
    sub = torch.utils.data.Subset(ds, idx.tolist())
    return torch.utils.data.DataLoader(sub, batch_size=batch_size, shuffle=False,
                                       num_workers=num_workers, pin_memory=True), len(sub)


def main():
    from torchvision import models
    ap = argparse.ArgumentParser()
    ap.add_argument("--val_root", required=True)
    ap.add_argument("--ratios_json", required=True)
    ap.add_argument("--out", default="resnet_imc_results.json")
    ap.add_argument("--n_images", type=int, default=2000)
    ap.add_argument("--subset_seed", type=int, default=42)
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--num_workers", type=int, default=4)
    ap.add_argument("--calib_batches", type=int, default=4)
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--drift_grid", default="1,1e2,1e4,1e6")
    ap.add_argument("--no_write_noise", action="store_true")
    ap.add_argument("--dtype", default="float32")
    ap.add_argument("--timing_only", action="store_true")
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dt = torch.float64 if args.dtype == "float64" else torch.float32
    drift_grid = [float(s) for s in args.drift_grid.split(",")]
    ratios = json.load(open(args.ratios_json))
    wn_on = not args.no_write_noise

    loader, nsub = imagenet_subset_loader(args.val_root, args.n_images, args.subset_seed,
                                          args.batch_size, args.num_workers)
    print(f"subset images: {nsub}  device: {device}  dtype: {dt}  batch: {args.batch_size}", flush=True)

    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).eval().to(dt).to(device)
    pcm_paths = [n[:-len(".weight")] for n, _ in model.named_parameters() if R.is_pcm_weight(n)]
    print("calibrating activation scales (float model) ...", flush=True)
    act_scales = float_calibrate(model, pcm_paths, loader, device, args.calib_batches)

    imc = R.build_imc_resnet(model, device=device, dtype=dt)
    arms = build_arms(imc, ratios, device)

    if args.timing_only:
        torch.cuda.synchronize() if device == "cuda" else None
        t0 = time.time()
        a1, a5 = run_config(model, imc, loader, device, dt, act_scales,
                            arms["layer_aware"], 1e6, 5, 0, wn_on)
        torch.cuda.synchronize() if device == "cuda" else None
        el = time.time() - t0
        print(f"[timing] one full-subset config (layer_aware, t=1e6, 5-bit ADC): "
              f"{el:.1f}s  =>  {el / nsub * 1000:.1f} ms/img  top1={a1:.4f}", flush=True)
        n_cfg = 3 + 3 * 2 * len(drift_grid) * args.seeds
        print(f"[timing] full sweep ~= {n_cfg} configs => est {el * n_cfg / 60:.0f} min "
              f"(clean is cheaper; this is an upper bound)", flush=True)
        return

    meta = {"model": "resnet50", "n_images": nsub, "subset_seed": args.subset_seed,
            "batch_size": args.batch_size, "seeds": args.seeds, "drift_grid": drift_grid,
            "write_noise": wn_on, "dtype": str(dt), "cellBit": R.CELLBIT,
            "n_mag_cells": R.NCELL, "adc": "5-bit (NeuroSim levelOutput=32)",
            "scheme": "2-bit sign-magnitude (III)",
            "sigma_cell": R.writenoise_sigma_cell(),
            "device": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
            "torch": torch.__version__}
    print("starting sweep ...", flush=True)
    t0 = time.time()
    runs = sweep(model, imc, loader, device, dt, act_scales, arms,
                 drift_grid, args.seeds, wn_on)
    meta["elapsed_sec"] = time.time() - t0
    json.dump({"meta": meta, "runs": runs}, open(args.out, "w"), indent=2)
    print(f"\nwrote {args.out}  ({meta['elapsed_sec']/60:.1f} min)", flush=True)


if __name__ == "__main__":
    main()


Writing run_resnet_imc.py


In [ ]:
# layer-aware allocation (Table IX) from your Drive
RATIOS = '/content/LayerAware_Ratios_ResNet50.json'
!cp {PROJECT}/outputs_ResNet50_ImageNet/LayerAware_Ratios_ResNet50.json {RATIOS}
import json; print('layers in allocation:', len(json.load(open(RATIOS))))

## 4. Timing probe — run FIRST to size the full run
Runs one config (layer-aware, t=1e6, 5-bit ADC) over a small 200-image subset and reports ms/image plus an estimate for the full sweep. If you hit out-of-memory, lower `--batch_size`.

In [7]:
!python run_resnet_imc.py --val_root {VAL_ROOT} --ratios_json {RATIOS} --n_images 200 --batch_size 32 --calib_batches 2 --drift_grid 1,1e2,1e4,1e6 --seeds 3 --timing_only

Traceback (most recent call last):
  File "/content/run_resnet_imc.py", line 25, in <module>
    import imc_resnet as R
  File "/content/imc_resnet.py", line 30, in <module>
    from imc_torch import imc_matmul_t
  File "/content/imc_torch.py", line 16, in <module>
    from imc_vec import load_model, read_hex_bytes, LAYERS, WCOUNT  # reuse numpy model loader
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'imc_vec'


## 5. Full sweep
3 arms (none / uniform 5% / layer-aware) x {no-ADC, 5-bit ADC} x drift-grid x seeds, on a fixed random subset. Output JSON is saved to your Drive. Adjust `--n_images`, `--seeds`, `--drift_grid` based on the timing probe (H100 can usually afford 2000-5000 images, 3 seeds).

In [ ]:
OUT = f'{PROJECT}/outputs_ResNet50_ImageNet/resnet_imc_results.json'
!python run_resnet_imc.py --val_root {VAL_ROOT} --ratios_json {RATIOS} --n_images 2000 --subset_seed 42 --batch_size 32 --calib_batches 4 --seeds 3 --drift_grid 1,1e2,1e4,1e6 --out {OUT}

## 6. Quick look at the result

In [ ]:
import json
d = json.load(open(OUT))
print('meta:', {k:d['meta'][k] for k in ['n_images','seeds','drift_grid','device','elapsed_sec']})
print('\nclean (full INT8, no drift/ADC/noise):')
for r in d['runs']:
    if r['kind']=='clean': print(f"  {r['arm']:12s} top1={r['top1']:.4f} top5={r['top5']:.4f}")
print('\n5-bit ADC, top1 mean+/-std:')
for r in d['runs']:
    if r['kind']=='sweep' and r['adc_bits']==5:
        print(f"  {r['arm']:12s} t={r['drift_t']:>8.0f}  {r['top1_mean']:.4f}+/-{r['top1_std']:.4f}")